In [ ]:
import os
from pathlib import Path

import pandas as pd
import time


def get_env_value(key, default=None, env_path=".env"):
    value = os.getenv(key)
    if value:
        return value

    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8") as env_file:
            for line in env_file:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                name, raw_value = line.split("=", 1)
                if name.strip() == key:
                    return raw_value.strip().strip('"').strip("'")
    return default


def get_env_int(key, default):
    return int(get_env_value(key, str(default)))


def get_env_float(key, default):
    return float(get_env_value(key, str(default)))


csv_path = Path(get_env_value("CSV_PATH", "IOT Data Simulation/smart_logistic_tracker_japan.csv"))
sample_rows = get_env_int("SAMPLE_ROWS", 5)
write_delay_seconds = get_env_float("WRITE_DELAY_SECONDS", 0.1)
write_gas_limit = get_env_int("WRITE_GAS_LIMIT", 3000000)
enable_duplicate_writes = get_env_value("ENABLE_DUPLICATE_WRITES", "false").lower() in {"1", "true", "yes", "on"}
abi_path = Path(get_env_value("ABI_PATH", "abi.json"))

# Load the CSV file with basic error handling
try:
    df = pd.read_csv(csv_path)
    print(f"Total records in CSV: {len(df)}")
    print(f"First {sample_rows} records:")

    # Display the first few rows
    display(df.head(sample_rows))
except FileNotFoundError:
    print(f"❌ CSV file not found: {csv_path}")
    df = pd.DataFrame()
except pd.errors.EmptyDataError:
    print(f"❌ CSV file is empty: {csv_path}")
    df = pd.DataFrame()
except pd.errors.ParserError as error:
    print(f"❌ Failed to parse CSV file {csv_path}: {error}")
    df = pd.DataFrame()
except Exception as error:
    print(f"❌ Unexpected error while loading {csv_path}: {error}")
    df = pd.DataFrame()

Total records in CSV: 100
First 5 records:


,timestamp,carrier,tracking_number,package_id,origin,current_location,delivery_location,prefecture,latitude,longitude,...,waiting_time_minutes,perishable,temperature,humidity,rfid_tag,rfid_verified,tamper_alert,traffic_status,inventory_level,asset_utilization
0,2026-05-04 13:50:26.857905,Yamato Transport,942646961460,PKG7545,Tokyo,Naha Central Post Office,Tokyo,Kanagawa,35.993159,139.038781,...,45,No,10.7,40,RFID736892,False,No,Heavy,99,84.59
1,2026-05-03 23:36:26.858095,Japan Post,74355111775,PKG2659,Tokyo,Nagoya Central Post Office,Kyoto,Kanagawa,35.691292,139.130870,...,54,Yes,6.2,82,RFID156229,False,Yes,Detour,363,53.39
2,2026-05-04 08:32:26.858217,Japan Post,217497030475,PKG7965,Osaka,Nagoya Central Post Office,Osaka,Aichi,35.591109,139.784940,...,144,Yes,-3.1,86,RFID890703,True,Yes,Heavy,25,95.75
3,2026-05-04 02:35:26.858332,Japan Post,249781996688,PKG5296,Fukuoka,Sapporo Central Post Office,Sapporo,Osaka,35.570440,139.689163,...,173,Yes,6.3,87,RFID603182,True,Yes,Detour,145,63.84
4,2026-05-04 08:28:26.858444,Japan Post,718415724062,PKG9987,Fukuoka,Yokohama Sales Office,Sapporo,Hokkaido,35.679375,139.408071,...,82,No,0.7,60,RFID921432,False,Yes,Detour,34,56.28


In [2]:
from web3 import Web3

# Connect to local blockchain
ganache_url = get_env_value("GANACHE_URL", "http://127.0.0.1:8545")
web3 = Web3(Web3.HTTPProvider(ganache_url))

# Verify connection
if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

✅ Connected to Ganache successfully!


In [3]:
import json

# Use the loaded ABI path and deployed contract address.
contract_address = get_env_value("CONTRACT_ADDRESS")
if not contract_address:
    raise ValueError("CONTRACT_ADDRESS is missing. Set it in .env or the environment.")
contract_address = Web3.to_checksum_address(contract_address)

# Load the ABI that matches the deployed contract in this repository
with open(abi_path, "r", encoding="utf-8") as abi_file:
    abi = json.load(abi_file)

# Load the smart contract
contract = web3.eth.contract(address=contract_address, abi=abi)

# Use the on-chain contract owner when available; fall back to an explicit override.
contract_owner = contract.functions.owner().call()
if contract_owner not in web3.eth.accounts:
    override_owner = get_env_value("CONTRACT_OWNER")
    if not override_owner:
        raise ValueError(
            f"Contract owner {contract_owner} is not unlocked in Ganache. "
            "Set CONTRACT_OWNER in .env to an unlocked account."
        )
    contract_owner = Web3.to_checksum_address(override_owner)
    if contract_owner not in web3.eth.accounts:
        raise ValueError(
            f"CONTRACT_OWNER {contract_owner} is not unlocked in Ganache."
        )

web3.eth.default_account = contract_owner

print(f"✅ Connected to Smart Contract at {contract_address}")
print(f"✅ Using sender account: {web3.eth.default_account}")

✅ Connected to Smart Contract at 0x7abf4b356FB67C8a9917c7E1E543895DB1Bf53b4
✅ Using sender account: 0x1C73Dd704ffeE88a4f4aAD5bA3B1af87C5884D0F


In [ ]:
def record_exists(package_id, data_type, data_value):
    """Return True when the exact record is already stored on-chain."""

    total_records = contract.functions.getTotalRecords().call()
    for record_index in range(total_records):
        record = contract.functions.getRecord(record_index).call()
        if (
            str(record[1]) == str(package_id)
            and str(record[2]) == str(data_type)
            and str(record[3]) == str(data_value)
        ):
            return True
    return False


def send_iot_data(package_id, data_type, data_value):
    """
    Sends logistics IoT data
    to the deployed smart contract
    """

    if not enable_duplicate_writes and record_exists(package_id, data_type, data_value):
        print(
            f"ℹ️ Skipped duplicate | {package_id} | "
            f"Type: {data_type} | Value: {data_value}"
        )
        return False

    if enable_duplicate_writes:
        print("ℹ️ Duplicate-write mode is ON; exact duplicates will be stored.")

    txn = contract.functions.storeData(
        package_id,
        data_type,
        data_value
    ).transact({
        'from': web3.eth.default_account,
        'gas': write_gas_limit
    })

    # Wait for transaction confirmation
    receipt = web3.eth.wait_for_transaction_receipt(txn)

    print(
        f"✅ Data Stored | {package_id} | "
        f"Type: {data_type} | "
        f"Value: {data_value} | "
        f"Txn Hash: {receipt.transactionHash.hex()}"
    )
    return True

# Each CSV row writes two contract entries: Location and Status
target_contract_records = int(get_env_value("TARGET_CONTRACT_RECORDS", "100"))
target_rows = target_contract_records // 2
current_records = contract.functions.getTotalRecords().call()
max_entries = contract.functions.MAX_ENTRIES().call()
remaining_entries = max_entries - current_records
rows_to_store = min(len(df), target_rows, remaining_entries // 2)

print(f"Target contract records: {target_contract_records}")
print(f"Current records: {current_records}")
print(f"Maximum records: {max_entries}")
print(f"Remaining contract slots: {remaining_entries}")
print(f"Rows that can still be stored safely: {rows_to_store}")
print(f"Duplicate writes enabled: {enable_duplicate_writes}")

if target_contract_records % 2 != 0:
    print("⚠️ TARGET_CONTRACT_RECORDS is odd, so the last record is ignored to keep row pairs complete.")

if rows_to_store <= 0:
    print("⚠️ No remaining storage capacity on the contract.")
else:
    stored_rows = 0
    skipped_rows = 0

    for index, row in df.head(rows_to_store).iterrows():
        package_id = str(row["package_id"])
        location = str(row["current_location"])
        status = str(row["latest_status"])

        location_stored = send_iot_data(package_id, "Location", location)
        status_stored = send_iot_data(package_id, "Status", status)

        if location_stored and status_stored:
            stored_rows += 1
        else:
            skipped_rows += 1

        # Delay between transactions
        time.sleep(write_delay_seconds)

    print(f"\n✅ Successfully stored {stored_rows} new rows on the blockchain!")
    if skipped_rows:
        print(f"ℹ️ Skipped {skipped_rows} duplicate rows.")

Target contract records: 100
Current records: 166
Maximum records: 500
Remaining contract slots: 334
Rows that can still be stored safely: 50
ℹ️ Skipped duplicate | PKG7545 | Type: Location | Value: Naha Central Post Office
ℹ️ Skipped duplicate | PKG7545 | Type: Status | Value: Out for Delivery
ℹ️ Skipped duplicate | PKG2659 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG2659 | Type: Status | Value: Arrival
ℹ️ Skipped duplicate | PKG7965 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG7965 | Type: Status | Value: Storage
ℹ️ Skipped duplicate | PKG5296 | Type: Location | Value: Sapporo Central Post Office
ℹ️ Skipped duplicate | PKG5296 | Type: Status | Value: Delivered to the delivery address
ℹ️ Skipped duplicate | PKG9987 | Type: Location | Value: Yokohama Sales Office
ℹ️ Skipped duplicate | PKG9987 | Type: Status | Value: Bring it back due to your absence
ℹ️ Skipped duplicate | PKG8392 | Type: Location | Value: Nagoya Cent

In [5]:
current_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {current_records}")

Total IoT records stored: 166


In [6]:
current_records = contract.functions.getTotalRecords().call()
max_entries = contract.functions.MAX_ENTRIES().call()
remaining_entries = max_entries - current_records

print(f"Current IoT records stored: {current_records}")
print(f"Maximum records allowed: {max_entries}")
print(f"Remaining storage slots: {remaining_entries}")

if remaining_entries == 0:
    print("⚠️ The contract is full. Redeploy a new contract or reset the chain to store more data.")
elif remaining_entries <= 20:
    print("⚠️ The contract is nearing capacity.")

Current IoT records stored: 166
Maximum records allowed: 500
Remaining storage slots: 334


In [7]:
# Retrieve and display the first stored record
first_record = contract.functions.getRecord(0).call()

print("📦 First Stored Record")
print(f"Timestamp: {first_record[0]}")
print(f"Package ID: {first_record[1]}")
print(f"Data Type: {first_record[2]}")
print(f"Data Value: {first_record[3]}")

📦 First Stored Record
Timestamp: 1780192485
Package ID: PKG7545
Data Type: Location
Data Value: Naha Central Post Office
